In [0]:
spark.sql("USE CATALOG workspace"); spark.sql("USE SCHEMA manufacturing")
for t in ["bronze_dim_machines","bronze_dim_products","bronze_mes_runs","bronze_downtime_events",
          "bronze_sensors","bronze_quality","silver_mes_runs","silver_run_oee","gold_oee_by_machine",
          "gold_daily_oee","gold_weekly_oee","gold_loss_taxonomy","gold_downtime_pareto",
          "gold_machine_health","gold_shift_performance","dim_machine_scd2"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("All old tables deleted. Starting fresh.")

All old tables deleted. Starting fresh.


In [0]:
from pyspark.sql import functions as F, Window
import random
from datetime import date, timedelta
random.seed(11)

LINES=["LINE-A","LINE-B","LINE-C","LINE-D","LINE-E"]; TYPES=["CNC","Press","Welder","Assembler","Lathe"]
LINE_BIAS={"LINE-A":0.0,"LINE-B":-0.14,"LINE-C":0.03,"LINE-D":0.09,"LINE-E":-0.05}
SHIFT_BIAS={"morning":0.0,"afternoon":-0.02,"night":-0.05}
LOSS={"Breakdown":("Availability","Unplanned",(30,180)),"Changeover":("Availability","Planned",(20,55)),
      "Planned Maintenance":("Availability","Planned",(30,90)),"Minor Stop":("Performance","Unplanned",(2,9)),
      "Reduced Speed":("Performance","Unplanned",(5,15)),"Material Starvation":("Availability","Unplanned",(10,40))}
RW={"Breakdown":0.12,"Changeover":0.25,"Planned Maintenance":0.06,"Minor Stop":0.35,"Reduced Speed":0.12,"Material Starvation":0.10}

MACHINES=[]
for i in range(1,51):
    line=random.choice(LINES); age=random.randint(1,18)
    health=max(0.30,min(1.0, 0.95+LINE_BIAS[line]-age*0.022+random.uniform(-0.07,0.07)))
    MACHINES.append({"machine_id":f"M{i:02d}","machine_name":f"MC-{i:02d}","line":line,
        "machine_type":random.choice(TYPES),"age_years":age,
        "maintenance_tier":"premium" if health>0.80 else "standard","health":health})
IDEAL={f"P{n}":c for n,c in [("100",12.0),("200",30.0),("300",45.0),("400",20.0),("500",18.0),("600",25.0)]}
PIDS=list(IDEAL)

runs=[]; events=[]; start=date.today()-timedelta(days=89); eid=0
for d in range(90):
    day=(start+timedelta(days=d)).isoformat()
    for m in MACHINES:
        h=m["health"]; prev=None
        for shift in ["morning","afternoon","night"]:
            pid=random.choice(PIDS); planned=480; dt=0; chg=(pid!=prev)
            n_events=max(0,int(random.gauss(1.5+(1-h)*5, 1.3)))
            for _ in range(n_events):
                reason=random.choices(list(RW),weights=list(RW.values()))[0]
                cat,ptype,(lo,hi)=LOSS[reason]; dur=random.randint(lo,hi)
                if reason=="Breakdown": dur=int(dur*(1.8-h*1.0))
                dt+=dur; events.append((eid,day,m["machine_id"],m["line"],shift,reason,cat,ptype,dur)); eid+=1
            if chg:
                dur=random.randint(20,55); dt+=dur
                events.append((eid,day,m["machine_id"],m["line"],shift,"Changeover","Availability","Planned",dur)); eid+=1
            dt=min(dt,int(planned*0.55)); rt=planned-dt
            perf=max(0.55,min(0.99,(0.83+h*0.14)+SHIFT_BIAS[shift]+random.uniform(-0.04,0.03)))
            qual=max(0.88,min(0.999,(0.955+h*0.04)+random.uniform(-0.02,0.005)))
            total=int((rt*60/IDEAL[pid])*perf); ss=int(total*0.02) if chg else 0
            good=max(0,int(total*qual)-ss)
            runs.append((day,m["machine_id"],pid,shift,planned,dt,total,good,ss)); prev=pid

runs_df=spark.createDataFrame(runs,["run_date","machine_id","product_id","shift","planned_min","downtime_min","total_units","good_units","startup_scrap"]).withColumn("run_date",F.to_date("run_date"))
events_df=spark.createDataFrame(events,["event_id","event_date","machine_id","line","shift","reason","loss_category","planned_type","duration_min"]).withColumn("event_date",F.to_date("event_date"))

spark.createDataFrame([{k:m[k] for k in ["machine_id","machine_name","line","machine_type","age_years","maintenance_tier"]} for m in MACHINES]).write.format("delta").mode("overwrite").saveAsTable("bronze_dim_machines")
spark.createDataFrame([(k,v) for k,v in IDEAL.items()],["product_id","ideal_cycle_time_sec"]).write.format("delta").mode("overwrite").saveAsTable("bronze_dim_products")
runs_df.write.format("delta").mode("overwrite").saveAsTable("bronze_mes_runs")
events_df.write.format("delta").mode("overwrite").saveAsTable("bronze_downtime_events")
print("Runs:",runs_df.count()," | Downtime events:",events_df.count())

Runs: 13500  | Downtime events: 42673


In [0]:
from pyspark.sql import Window

cols=["udi","pid_ai","type","air_k","proc_k","rot","torque","wear","fail","twf","hdf","pwf","osf","rnf"]
ai4i=spark.read.option("header",True).option("inferSchema",True).csv("/Volumes/workspace/manufacturing/seed/ai4i2020.csv").toDF(*cols)

# health lookup, and rank AI4I readings by tool-wear severity
health_df=spark.createDataFrame([(m["machine_id"],round(m["health"],3)) for m in MACHINES],["machine_id","health"])
ai4i_ranked=ai4i.withColumn("wear_pct",F.percent_rank().over(Window.orderBy("wear")))

# assign readings so LESS healthy machines get MORE severe (higher-wear) readings
rand_m=F.concat(F.lit("M"),F.lpad((F.floor(F.rand()*50)+1).cast("int").cast("string"),2,"0"))
sensors=(ai4i_ranked.crossJoin(spark.range(200))
    .withColumn("machine_id",rand_m).join(health_df,"machine_id","left")
    .withColumn("keep",F.when(F.abs(F.col("wear_pct")-(1-F.col("health")))<0.35,F.lit(1))
                        .otherwise(F.when(F.rand()<0.15,F.lit(1)).otherwise(F.lit(0))))
    .filter("keep=1")
    .withColumn("temperature_c",F.round(F.col("proc_k")-273.15,1))
    .withColumn("rotational_speed_rpm",F.col("rot").cast("int"))
    .withColumn("torque_nm",F.round(F.col("torque"),1))
    .withColumn("tool_wear_min",F.col("wear").cast("int"))
    .withColumn("machine_failure",F.col("fail").cast("int"))
    .withColumn("reading_date",F.date_sub(F.current_date(),(F.rand()*90).cast("int")))
    .select("machine_id","reading_date","temperature_c","rotational_speed_rpm","torque_nm","tool_wear_min","machine_failure"))
sensors.write.format("delta").mode("overwrite").saveAsTable("bronze_sensors")

# quality (unchanged — fail rate tied to health)
mh=spark.createDataFrame([(m["machine_id"],round(1-m["health"],3)) for m in MACHINES],["machine_id","risk"])
q=(spark.range(200000).withColumn("machine_id",rand_m).join(mh,"machine_id","left")
   .withColumn("result",F.when(F.rand()<F.col("risk")*0.30+0.01,"FAIL").otherwise("PASS"))
   .withColumn("inspected_date",F.date_sub(F.current_date(),(F.rand()*90).cast("int")))
   .select("machine_id","result","inspected_date"))
q.write.format("delta").mode("overwrite").saveAsTable("bronze_quality")
print("Sensors:",sensors.count()," | Quality:",q.count())
# verify tool wear now VARIES per machine:
display(sensors.groupBy("machine_id").agg(F.round(F.avg("tool_wear_min"),1).alias("avg_wear")).orderBy(F.desc("avg_wear")))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Sensors: 1286647  | Quality: 200000


machine_id,avg_wear
M03,120.2
M30,117.4
M40,109.7
M34,106.9
M25,104.4
M05,91.4
M46,88.0
M48,87.7
M42,85.6
M47,84.7


In [0]:
valid=(F.col("product_id").isNotNull()&(F.col("total_units")>0)&(F.col("good_units")<=F.col("total_units")))
spark.table("bronze_mes_runs").filter(valid).write.format("delta").mode("overwrite").saveAsTable("silver_mes_runs")
r=(spark.table("silver_mes_runs").join(spark.table("bronze_dim_products"),"product_id","left")
   .withColumn("run_time_min",F.col("planned_min")-F.col("downtime_min"))
   .withColumn("availability",F.col("run_time_min")/F.col("planned_min"))
   .withColumn("performance",F.least((F.col("ideal_cycle_time_sec")*F.col("total_units"))/(F.col("run_time_min")*60.0),F.lit(1.0)))
   .withColumn("quality",F.col("good_units")/F.col("total_units"))
   .withColumn("oee",F.col("availability")*F.col("performance")*F.col("quality"))
   .withColumn("scrap_units",F.col("total_units")-F.col("good_units")))
r.write.format("delta").mode("overwrite").saveAsTable("silver_run_oee")
print("Silver OEE written.")

Silver OEE written.


In [0]:
r=spark.table("silver_run_oee")
(r.groupBy("machine_id").agg(
    F.round(F.avg("availability"),3).alias("availability"),F.round(F.avg("performance"),3).alias("performance"),
    F.round(F.avg("quality"),3).alias("quality"),F.round(F.avg("oee"),3).alias("oee"),
    F.sum("downtime_min").alias("total_downtime_min"),
    F.round(F.sum("scrap_units")/F.sum("total_units"),3).alias("scrap_rate"))
   .join(spark.table("bronze_dim_machines"),"machine_id","left")
   .write.format("delta").mode("overwrite").saveAsTable("gold_oee_by_machine"))

daily=(r.groupBy("machine_id","run_date").agg(F.round(F.avg("oee"),3).alias("daily_oee"),F.sum("downtime_min").alias("downtime_min")))
w7=Window.partitionBy("machine_id").orderBy("run_date").rowsBetween(-6,0)
daily.withColumn("rolling_7d_oee",F.round(F.avg("daily_oee").over(w7),3)).write.format("delta").mode("overwrite").saveAsTable("gold_daily_oee")
(daily.withColumn("year",F.year("run_date")).withColumn("week",F.weekofyear("run_date"))
  .groupBy("year","week").agg(F.round(F.avg("daily_oee"),3).alias("weekly_oee")).orderBy("year","week")
  .write.format("delta").mode("overwrite").saveAsTable("gold_weekly_oee"))
(r.groupBy("shift").agg(F.round(F.avg("oee"),3).alias("oee"),F.round(F.avg("availability"),3).alias("availability"),
    F.round(F.avg("performance"),3).alias("performance")).write.format("delta").mode("overwrite").saveAsTable("gold_shift_performance"))
print("Gold OEE marts written.")

Gold OEE marts written.


In [0]:
ev=spark.table("bronze_downtime_events")
(ev.groupBy("loss_category","planned_type").agg(
    F.sum("duration_min").alias("total_downtime_min"),F.count("*").alias("event_count"))
   .orderBy(F.desc("total_downtime_min"))
   .write.format("delta").mode("overwrite").saveAsTable("gold_loss_taxonomy"))
reason=(ev.groupBy("reason").agg(F.sum("duration_min").alias("total_min"),F.count("*").alias("events")))
tot=reason.agg(F.sum("total_min")).collect()[0][0]
wP=Window.orderBy(F.desc("total_min")).rowsBetween(Window.unboundedPreceding,0)
(reason.withColumn("pct_of_total",F.round(F.col("total_min")/tot*100,1))
    .withColumn("cumulative_pct",F.round(F.sum("total_min").over(wP)/tot*100,1))
    .orderBy(F.desc("total_min"))
    .write.format("delta").mode("overwrite").saveAsTable("gold_downtime_pareto"))
print("Loss taxonomy + Pareto written.")
display(spark.table("gold_downtime_pareto"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Loss taxonomy + Pareto written.


reason,total_min,events,pct_of_total,cumulative_pct
Changeover,736980,19670,50.8,50.8
Breakdown,427220,3690,29.5,80.3
Planned Maintenance,113095,1895,7.8,88.1
Material Starvation,75421,3032,5.2,93.3
Minor Stop,58849,10646,4.1,97.4
Reduced Speed,37770,3740,2.6,100.0


In [0]:
sm=(spark.table("bronze_sensors").groupBy("machine_id").agg(F.round(F.avg("temperature_c"),1).alias("avg_temp_c"),F.round(F.avg("tool_wear_min"),1).alias("avg_tool_wear")))
qm=(spark.table("bronze_quality").groupBy("machine_id").agg(F.round(F.avg(F.when(F.col("result")=="FAIL",1.0).otherwise(0.0)),3).alias("fail_rate")))
(spark.table("gold_oee_by_machine").select("machine_id","machine_name","line","age_years","oee","availability","performance","quality","scrap_rate","total_downtime_min")
  .join(sm,"machine_id","left").join(qm,"machine_id","left")
  .write.format("delta").mode("overwrite").saveAsTable("gold_machine_health"))
h=spark.table("gold_machine_health")
print("OEE vs scrap corr:",round(h.stat.corr("oee","scrap_rate"),3)," | OEE vs fail corr:",round(h.stat.corr("oee","fail_rate"),3))

if not spark.catalog.tableExists("dim_machine_scd2"):
    (spark.table("bronze_dim_machines").withColumn("valid_from",F.current_date()).withColumn("valid_to",F.lit(None).cast("date")).withColumn("is_current",F.lit(True)).write.format("delta").saveAsTable("dim_machine_scd2"))
upd=spark.table("bronze_dim_machines").withColumn("maintenance_tier",F.when(F.col("machine_id")=="M01",F.lit("premium")).otherwise(F.col("maintenance_tier")))
upd.createOrReplaceTempView("mu")
spark.sql("MERGE INTO dim_machine_scd2 t USING mu u ON t.machine_id=u.machine_id AND t.is_current=true WHEN MATCHED AND t.maintenance_tier<>u.maintenance_tier THEN UPDATE SET t.is_current=false,t.valid_to=current_date()")
spark.sql("INSERT INTO dim_machine_scd2 SELECT u.*,current_date(),CAST(NULL AS date),true FROM mu u JOIN dim_machine_scd2 t ON u.machine_id=t.machine_id AND t.is_current=false AND t.valid_to=current_date()")
print("Machine health + SCD2 done.")
display(spark.table("gold_oee_by_machine").orderBy("oee").select("machine_id","line","age_years","availability","performance","quality","oee","scrap_rate"))

OEE vs scrap corr: -0.987  | OEE vs fail corr: -0.991
Machine health + SCD2 done.


machine_id,line,age_years,availability,performance,quality,oee,scrap_rate
M03,LINE-B,17,0.672,0.863,0.947,0.55,0.053
M30,LINE-B,15,0.676,0.862,0.947,0.552,0.053
M40,LINE-B,13,0.699,0.87,0.949,0.578,0.05
M25,LINE-B,14,0.701,0.875,0.95,0.583,0.05
M34,LINE-B,16,0.717,0.872,0.95,0.595,0.05
M05,LINE-A,18,0.725,0.884,0.954,0.611,0.047
M42,LINE-B,8,0.734,0.888,0.955,0.623,0.045
M47,LINE-A,12,0.74,0.891,0.957,0.631,0.043
M48,LINE-C,17,0.746,0.886,0.955,0.631,0.046
M46,LINE-C,17,0.747,0.886,0.953,0.631,0.046


In [0]:
 display(spark.table("gold_machine_health"))

machine_id,machine_name,line,age_years,oee,availability,performance,quality,scrap_rate,total_downtime_min,avg_temp_c,avg_tool_wear,fail_rate
M09,MC-09,LINE-E,1,0.747,0.831,0.931,0.965,0.035,21955,36.9,108.1,0.036
M15,MC-15,LINE-D,4,0.787,0.863,0.941,0.97,0.03,17797,36.9,107.9,0.012
M19,MC-19,LINE-D,3,0.776,0.854,0.939,0.969,0.03,18977,36.9,107.9,0.018
M20,MC-20,LINE-C,11,0.646,0.754,0.894,0.958,0.042,31879,36.9,108.3,0.114
M29,MC-29,LINE-E,4,0.674,0.776,0.906,0.96,0.04,29089,36.9,107.7,0.081
M35,MC-35,LINE-D,6,0.754,0.839,0.929,0.967,0.032,20910,36.9,108.1,0.032
M38,MC-38,LINE-D,3,0.775,0.856,0.935,0.968,0.031,18708,36.9,108.5,0.023
M42,MC-42,LINE-B,8,0.623,0.734,0.888,0.955,0.045,34413,36.9,108.2,0.116
M43,MC-43,LINE-E,3,0.711,0.81,0.913,0.962,0.038,24688,36.9,108.3,0.066
M12,MC-12,LINE-C,14,0.676,0.779,0.905,0.959,0.041,28608,36.9,108.0,0.088


In [0]:
display(spark.table("gold_downtime_pareto"))

reason,total_min,events,pct_of_total,cumulative_pct
Changeover,736980,19670,50.8,50.8
Breakdown,427220,3690,29.5,80.3
Planned Maintenance,113095,1895,7.8,88.1
Material Starvation,75421,3032,5.2,93.3
Minor Stop,58849,10646,4.1,97.4
Reduced Speed,37770,3740,2.6,100.0


In [0]:
display(spark.table("gold_loss_taxonomy"))

loss_category,planned_type,total_downtime_min,event_count
Availability,Planned,850075,21565
Availability,Unplanned,502641,6722
Performance,Unplanned,96619,14386


In [0]:
display(spark.table("gold_shift_performance"))

shift,oee,availability,performance
afternoon,0.69,0.787,0.91
night,0.668,0.788,0.88
morning,0.689,0.773,0.929


In [0]:
import mlflow, mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

features=["air_temp_k","process_temp_k","rot_speed","torque","tool_wear"]
X=pdf[features]; y=pdf["machine_failure"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=42)

mlflow.sklearn.autolog()
with mlflow.start_run(run_name="predictive_maintenance_rf"):
    rf=RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,
                              class_weight="balanced",random_state=42)
    rf.fit(Xtr,ytr)
    proba=rf.predict_proba(Xte)[:,1]; pred=rf.predict(Xte)

    auc      = roc_auc_score(yte, proba)
    precision= precision_score(yte, pred)
    recall   = recall_score(yte, pred)
    f1       = f1_score(yte, pred)
    cv       = cross_val_score(rf, X, y, cv=5, scoring="roc_auc")

    for k,v in {"test_auc":auc,"precision":precision,"recall":recall,"f1":f1}.items():
        mlflow.log_metric(k,v)

    print("=== Predictive Maintenance Model — Failure Detection ===")
    print(f"  ROC-AUC:      {auc:.3f}   (how well it separates fail vs OK — main metric)")
    print(f"  Recall:       {recall:.3f}   (% of real failures caught)")
    print(f"  Precision:    {precision:.3f}   (% of flagged machines that truly fail)")
    print(f"  F1-score:     {f1:.3f}")
    print(f"  5-fold CV AUC:{cv.mean():.3f} (+/- {cv.std():.3f})   (stable = not overfit)")
    print("\n  Top failure predictors:")
    for f,imp in sorted(zip(features,rf.feature_importances_),key=lambda x:-x[1])[:3]:
        print(f"    {f:16s} {imp:.3f}")

2026/08/05 02:19:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/05 02:19:40 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/ml

=== Predictive Maintenance Model — Failure Detection ===
  ROC-AUC:      0.971   (how well it separates fail vs OK — main metric)
  Recall:       0.788   (% of real failures caught)
  Precision:    0.500   (% of flagged machines that truly fail)
  F1-score:     0.612
  5-fold CV AUC:0.933 (+/- 0.048)   (stable = not overfit)

  Top failure predictors:
    torque           0.342
    rot_speed        0.313
    tool_wear        0.218


In [0]:
from pyspark.sql import functions as F

# Use the ORIGINAL AI4I (~10k rows) from your Volume for training — NOT the replicated 2M sensor table
cols=["udi","product_id","type","air_temp_k","process_temp_k","rot_speed","torque","tool_wear",
      "machine_failure","twf","hdf","pwf","osf","rnf"]
ai4i=spark.read.option("header",True).option("inferSchema",True).csv(
      "/Volumes/workspace/manufacturing/seed/ai4i2020.csv").toDF(*cols)

pdf = ai4i.select("air_temp_k","process_temp_k","rot_speed","torque","tool_wear","machine_failure").toPandas()
print("Rows:", len(pdf), "| Failure rate: %.1f%%" % (pdf.machine_failure.mean()*100))
print("Failures:", int(pdf.machine_failure.sum()))
pdf.head()

Rows: 10000 | Failure rate: 3.4%
Failures: 339


,air_temp_k,process_temp_k,rot_speed,torque,tool_wear,machine_failure
0,298.1,308.6,1551,42.8,0,0
1,298.2,308.7,1408,46.3,3,0
2,298.1,308.5,1498,49.4,5,0
3,298.2,308.6,1433,39.5,7,0
4,298.2,308.7,1408,40.0,9,0


In [0]:
# Are there duplicate rows inflating the score?
print("Total rows:", len(pdf), "| Unique rows:", len(pdf.drop_duplicates()))
# Did any leaky columns sneak in? (features should ONLY be sensors)
print("Features used:", features)

Total rows: 10000 | Unique rows: 10000
Features used: ['air_temp_k', 'process_temp_k', 'rot_speed', 'torque', 'tool_wear']


In [0]:
from pyspark.sql import Window

In [0]:
from pyspark.sql import functions as F

# Condition-based failure risk: derived from each machine's real state
# (older + lower OEE + more failures = higher risk)
risk=(spark.table("gold_machine_health")
  .withColumn("failure_risk", F.round(
      0.5*(1-F.col("oee")) + 0.3*(F.col("age_years")/18) + 0.2*F.col("fail_rate"), 3))
  .select("machine_id","line","age_years","oee","fail_rate","total_downtime_min","failure_risk")
  .orderBy(F.desc("failure_risk")))

risk.write.format("delta").mode("overwrite").saveAsTable("gold_failure_risk")
print("Failure risk index created.")
display(risk)

Failure risk index created.


machine_id,line,age_years,oee,fail_rate,total_downtime_min,failure_risk
M03,LINE-B,17,0.55,0.166,42449,0.542
M05,LINE-A,18,0.611,0.129,35650,0.52
M30,LINE-B,15,0.552,0.175,41979,0.509
M34,LINE-B,16,0.595,0.162,36627,0.502
M46,LINE-C,17,0.631,0.124,32811,0.493
M48,LINE-C,17,0.631,0.119,32867,0.492
M01,LINE-D,18,0.662,0.106,30308,0.49
M02,LINE-D,17,0.651,0.094,31833,0.477
M25,LINE-B,14,0.583,0.158,38727,0.473
M40,LINE-B,13,0.578,0.16,38954,0.46
